In [2]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from flaml import AutoML
from sklearn.model_selection import train_test_split

In [3]:
data = pd.read_csv('processed_data.csv')
data.head()

,Unnamed: 0,material,stroke,osc_freq,force,cycle,coe,Absolute_Energy,Amplitude,ASL,Average_Frequency,Initiation_Frequency,Parametric_1,Parametric_2,Peak_Frequency,Reverberation_Frequency,Threshold,stage
0,0,steel-glass,100,5,5,1,0.021518,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,stage1
1,1,steel-glass,100,5,5,2,0.009959,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,stage1
2,2,steel-glass,100,5,5,3,0.021819,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,stage1
3,3,steel-glass,100,5,5,4,0.019355,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,stage1
4,4,steel-glass,100,5,5,5,0.012795,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,stage1


In [ ]:
# separate features and labels
X = data[['coe', 'Absolute_Energy', 'Amplitude', 'ASL', 'Average_Frequency', 'Initiation_Frequency', 'Parametric_1', 'Parametric_2', 'Peak_Frequency', 'Reverberation_Frequency', 'Threshold']] # features
y = data['stage']               # labels

In [5]:
# split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [6]:
# 初始化AutoML
automl = AutoML()

# 配置参数
settings = {
    "metric": 'accuracy',        # 评估指标
    "task": 'classification',    # 任务类型
    "log_file_name": 'tribology+AI.log',  # 日志文件
    "eval_method": 'cv',         # 使用交叉验证
    "split_ratio": 0.2,          # 验证集比例（当eval_method为'holdout'时）
    "n_splits": 5,               # 交叉验证折数
    "verbose": 3,                # 日志详细程度
}

# 训练模型
automl.fit(X_train=X_train, y_train=y_train, **settings)

[flaml.automl.logger: 05-09 00:34:10] {1728} INFO - task = classification
[flaml.automl.logger: 05-09 00:34:10] {1739} INFO - Evaluation method: cv
[flaml.automl.logger: 05-09 00:34:10] {1838} INFO - Minimizing error metric: 1-accuracy
[flaml.automl.logger: 05-09 00:34:10] {1861} WARNING - No search budget is provided via time_budget or max_iter. Training only one model per estimator. Zero-shot AutoML is used for certain tasks and estimators. To tune hyperparameters for each estimator, please provide budget either via time_budget or max_iter.
[flaml.automl.logger: 05-09 00:34:10] {1955} INFO - List of ML learners in AutoML Run: ['rf', 'lgbm', 'xgboost', 'extra_tree', 'xgb_limitdepth', 'sgd', 'lrl1']
[flaml.automl.logger: 05-09 00:34:10] {2258} INFO - iteration 0, current learner rf
[flaml.automl.logger: 05-09 00:34:13] {2393} INFO - Estimated sufficient time budget=10000s. Estimated necessary time budget=10s.
[flaml.automl.logger: 05-09 00:34:13] {2442} INFO -  at 3.3s,	estimator rf's 

In [7]:
# 打印最佳模型信息
print("\n最佳模型:", automl.best_estimator)
print("最佳参数配置:", automl.best_config)
print("最佳验证分数:", 1 - automl.best_loss)  # 因为FLAML最小化1-metric

# 在测试集上评估
y_pred = automl.predict(X_test)
test_accuracy = automl.score(X_test, y_test)
print("\n测试集准确率:", test_accuracy)

# 分类报告
from sklearn.metrics import classification_report
print("\n分类报告:")
print(classification_report(y_test, y_pred))


最佳模型: rf
最佳参数配置: {'n_estimators': 632, 'max_features': np.float64(1.0), 'max_leaves': 1360, 'criterion': 'entropy'}
最佳验证分数: 0.9422169811320755

测试集准确率: 0.9095477386934674

分类报告:
              precision    recall  f1-score   support

      stage1       1.00      1.00      1.00        67
      stage2       0.89      0.89      0.89        27
      stage3       0.44      0.44      0.44        16
      stage4       0.93      0.93      0.93        89

    accuracy                           0.91       199
   macro avg       0.81      0.81      0.81       199
weighted avg       0.91      0.91      0.91       199

